### Flipping DLC tracking files (left-right normalization)

This notebook standardizes the orientation of DeepLabCut tracking data across animals by ensuring that a given stimulus is always located on the same side (left or right).

Written by Anna Teruel-Sanchis


This first block of code flips the coordinates directly from the DLC h5s files. You should run this code before applying keypointmoseq model to the dlc data. 

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

INDEX_CSV = '/Users/annateruel/Desktop/wanhui/index.csv'
H5_DIR    = '/Users/annateruel/Desktop/wanhui/raw_tracking'
OUT_DIR   = '/Users/annateruel/Desktop/raw_flipped/'
VIDEO_DIR = '/Users/annateruel/Desktop/wanhui/raw_tracking'

TARGET_NA_SIDE = "L"

BODYPART_TO_PLOT = "snout"
MAX_FRAMES_PLOT  = None

def find_matching_video_for_h5(h5_path, video_dir):
    """
    Match DLC h5 -> mp4 by taking the h5 basename before 'DLC_'.
    Example:
      xxxDLC_..._filtered.h5  ->  xxx.mp4  (or xxx*.mp4)
    """
    base = os.path.basename(h5_path)
    stem = os.path.splitext(base)[0]

    if "DLC_" in stem:
        prefix = stem.split("DLC_")[0].rstrip("_- ")
    else:
        prefix = stem

    # try common patterns
    candidates = sorted(glob.glob(os.path.join(video_dir, prefix + "*.mp4")))
    if candidates:
        return candidates[0]

    # fallback: exact name mp4
    exact = os.path.join(video_dir, prefix + ".mp4")
    if os.path.exists(exact):
        return exact

    return None
def get_video_size(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {video_path}")
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    if W <= 0 or H <= 0:
        raise ValueError(f"Bad video dimensions for {video_path}: W={W}, H={H}")
    return W, H
def get_all_x_columns(df):
    """Return MultiIndex columns that correspond to x coordinates (any bodypart)."""
    if isinstance(df.columns, pd.MultiIndex):
        # assume coord is last level OR at least contains 'x'
        # robust: find any level with 'x'/'y'
        coord_level = None
        for lvl in range(df.columns.nlevels):
            vals = set(str(v).lower() for v in df.columns.get_level_values(lvl))
            if "x" in vals and "y" in vals:
                coord_level = lvl
                break
        if coord_level is None:
            raise ValueError("Could not find coord level containing x/y.")
        coord_vals = df.columns.get_level_values(coord_level).astype(str).str.lower()
        return df.columns[coord_vals == "x"]
    else:
        return [c for c in df.columns if str(c).lower().endswith("x")]
def flip_lr_dataframe_true(df, frame_width):
    """
    True left-right flip around the camera frame:
      x_flipped = (W - 1) - x
    """
    df_flipped = df.copy()
    x_cols = get_all_x_columns(df_flipped)
    if len(x_cols) == 0:
        raise ValueError("No x-columns found to flip.")
    df_flipped.loc[:, x_cols] = (frame_width - 1) - df_flipped.loc[:, x_cols]
    return df_flipped
def get_xy_columns_for_bodypart(df, bodypart):
    if isinstance(df.columns, pd.MultiIndex):
        # Find bodypart and coord levels robustly
        bp_level = None
        for lvl in range(df.columns.nlevels):
            if bodypart in df.columns.get_level_values(lvl):
                bp_level = lvl
                break
        if bp_level is None:
            raise ValueError(f"Bodypart '{bodypart}' not found.")

        coord_level = None
        for lvl in range(df.columns.nlevels):
            vals = set(str(v).lower() for v in df.columns.get_level_values(lvl))
            if "x" in vals and "y" in vals:
                coord_level = lvl
                break
        if coord_level is None:
            raise ValueError("Could not find coord level containing x/y.")

        cols = df.columns
        body_vals = cols.get_level_values(bp_level)
        coord_vals = cols.get_level_values(coord_level).astype(str).str.lower()

        x_cols = cols[(body_vals == bodypart) & (coord_vals == "x")]
        y_cols = cols[(body_vals == bodypart) & (coord_vals == "y")]

        if len(x_cols) == 0 or len(y_cols) == 0:
            raise ValueError(f"Could not find x/y for bodypart '{bodypart}'")
        return x_cols[0], y_cols[0]

    else:
        x_name = f"{bodypart}_x"
        y_name = f"{bodypart}_y"
        if x_name not in df.columns or y_name not in df.columns:
            raise ValueError(f"Could not find columns {x_name}, {y_name}.")
        return x_name, y_name
def plot_before_after(df_orig, df_flipped, bodypart, max_frames, title, out_path,
                      frame_width, frame_height):
    x_col, y_col = get_xy_columns_for_bodypart(df_orig, bodypart)
    n = len(df_orig) if max_frames is None else min(len(df_orig), max_frames)

    x0 = df_orig.loc[: n - 1, x_col].to_numpy()
    y0 = df_orig.loc[: n - 1, y_col].to_numpy()
    x1 = df_flipped.loc[: n - 1, x_col].to_numpy()
    y1 = df_flipped.loc[: n - 1, y_col].to_numpy()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)

    axes[0].plot(x0, y0, linewidth=0.5)
    axes[0].set_title("Original")

    axes[1].plot(x1, y1, linewidth=0.5)
    axes[1].set_title("Flipped (true W)")

    # video-like coordinates + fixed arena limits
    for ax in axes:
        ax.set_xlim(0, frame_width)
        ax.set_ylim(frame_height, 0)  # y-down
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("x")
        ax.set_ylabel("y")

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    plots_dir = os.path.join(OUT_DIR, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    info = pd.read_csv(INDEX_CSV)

    for _, row in info.iterrows():
        name = str(row["name"])
        na_side = str(row["NA"]).strip().upper()

        in_path = os.path.join(H5_DIR, name if name.endswith(".h5") else name + ".h5")
        if not os.path.exists(in_path):
            print(f"[WARN] File not found, skipping: {in_path}")
            continue

        # Find matching video and its size
        vid_path = find_matching_video_for_h5(in_path, VIDEO_DIR)
        if vid_path is None:
            print(f"[WARN] No matching video for: {os.path.basename(in_path)} (skipping flip)")
            continue

        try:
            W, H = get_video_size(vid_path)
        except Exception as e:
            print(f"[WARN] Could not read video size for {os.path.basename(vid_path)}: {e}")
            continue

        print(f"\nProcessing: {os.path.basename(in_path)}")
        print(f"  Video: {os.path.basename(vid_path)}  (W,H)=({W},{H})")

        df = pd.read_hdf(in_path)

        if na_side == TARGET_NA_SIDE:
            df_out = df
            flipped = False
        else:
            df_out = flip_lr_dataframe_true(df, frame_width=W)
            flipped = True

        out_path = os.path.join(OUT_DIR, os.path.basename(in_path))
        df_out.to_hdf(out_path, key="df", mode="w")
        print(f"  Saved to: {out_path} ({'flipped' if flipped else 'copied'})")

        if flipped and BODYPART_TO_PLOT is not None:
            try:
                plot_path = os.path.join(
                    plots_dir,
                    os.path.splitext(os.path.basename(in_path))[0] + "_traj.png",
                )
                plot_before_after(
                    df, df_out,
                    bodypart=BODYPART_TO_PLOT,
                    max_frames=MAX_FRAMES_PLOT,
                    title=os.path.basename(in_path),
                    out_path=plot_path,
                    frame_width=W,
                    frame_height=H,
                )
                print(f"  Plot saved to: {plot_path}")
            except Exception as e:
                print(f"  [WARN] Could not plot trajectories: {e}")

    print("\nDone.")
if __name__ == "__main__":
    main()

Just plotting the tracking data: 

In [ ]:
import os
import glob
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_DIR = "/Users/annateruel/Desktop/raw_flipped"
OUTPUT_DIR = "/Users/annateruel/Desktop"

BODY_PART = "snout"   # change to "nose", "tailbase", etc.
ALPHA = 0.6
LINEWIDTH = 1.2
DPI = 300

os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_dlc_h5(h5_path):
    """Load DLC h5 file into DataFrame"""
    return pd.read_hdf(h5_path)
def extract_xy(df, bodypart):
    """
    Robustly extract x,y for a bodypart from a DLC-style MultiIndex column DF.
    Works even if MultiIndex level order differs.
    """
    if not isinstance(df.columns, pd.MultiIndex):
        raise ValueError("Expected df.columns to be a MultiIndex (DLC .h5).")

    # --- Find which level contains the bodypart name ---
    bodypart_level = None
    for lvl in range(df.columns.nlevels):
        if bodypart in df.columns.get_level_values(lvl):
            bodypart_level = lvl
            break
    if bodypart_level is None:
        raise KeyError(
            f"Bodypart '{bodypart}' not found. Available examples: "
            f"{list(pd.unique(df.columns.get_level_values(1)))[:10]}"
        )

    # Select only columns for this bodypart (keep remaining levels)
    bp = df.loc[:, pd.IndexSlice[tuple(
        bodypart if i == bodypart_level else slice(None)
        for i in range(df.columns.nlevels)
    )]]

    # --- Find which remaining level contains x/y ---
    coord_level_in_bp = None
    for lvl in range(bp.columns.nlevels):
        vals = set(bp.columns.get_level_values(lvl))
        if ("x" in vals) and ("y" in vals):
            coord_level_in_bp = lvl
            break
    if coord_level_in_bp is None:
        raise KeyError(
            f"Could not find coord level with 'x'/'y' for bodypart '{bodypart}'. "
            f"Levels: {[list(pd.unique(bp.columns.get_level_values(l)))[:10] for l in range(bp.columns.nlevels)]}"
        )

    # Slice out x/y
    x = bp.xs("x", level=coord_level_in_bp, axis=1).to_numpy().squeeze()
    y = bp.xs("y", level=coord_level_in_bp, axis=1).to_numpy().squeeze()

    return x, y
def plot_trajectory(x, y, title, save_path):
    plt.figure(figsize=(5, 5))
    plt.plot(x, y, color="black", lw=LINEWIDTH, alpha=ALPHA)
    plt.scatter(x[0], y[0], s=20, c="green", label="start")
    plt.scatter(x[-1], y[-1], s=20, c="red", label="end")
    
    # plt.gca().invert_yaxis()  # DLC pixel coordinates
    plt.axis("equal")
    plt.axis("off")
    plt.title(title, fontsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI)
    plt.close()


h5_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*filtered.h5")))

print(f"Found {len(h5_files)} filtered files")

for h5_path in h5_files:
    fname = os.path.basename(h5_path)
    stem = fname.replace(".h5", "")

    print(f"Processing: {fname}")

    try:
        df = load_dlc_h5(h5_path)
        x, y = extract_xy(df, BODY_PART)

        save_path = os.path.join(
            OUTPUT_DIR,
            f"{stem}_{BODY_PART}_trajectory.png"
        )

        plot_trajectory(
            x,
            y,
            title=stem,
            save_path=save_path
        )

    except Exception as e:
        print(f"  ⚠️ Failed {fname}: {e}")
print("Done.")

Plotting tracking video to confirm tracking position

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import cv2

DLC_DIR   = "/Users/annateruel/Desktop/interpolated"
VIDEO_DIR = "/Users/annateruel/Desktop/interpolated"   # change if videos are elsewhere
OUT_DIR   = os.path.join(DLC_DIR, "overlay_videos")

BODYPART = "snout"
RADIUS = 5
TRAIL = True
TRAIL_MAXLEN = None        # set None for full trail (can be heavy)
TRAIL_STRIDE = 2          # draw every Nth point in the trail (speed)
CONF_THRESH = None        # e.g. 0.2 if you have likelihood; None = ignore

os.makedirs(OUT_DIR, exist_ok=True)
def find_matching_video(h5_path, video_dir):
    """
    Try to find a video that matches the DLC h5 name.
    Strategy: use the part before 'DLC_' if present, otherwise basename.
    """
    base = os.path.basename(h5_path)
    stem = base.replace(".h5", "")

    if "DLC_" in stem:
        vid_stem = stem.split("DLC_")[0].rstrip("_")
    else:
        vid_stem = stem

    # Try common extensions
    candidates = []
    for ext in (".mp4", ".avi", ".mov", ".m4v"):
        candidates += glob.glob(os.path.join(video_dir, vid_stem + "*" + ext))

    return sorted(candidates)[0] if candidates else None
def extract_xy_conf(df, bodypart):
    """
    Robust x/y (+ optional likelihood) extraction for a DLC MultiIndex df.
    """
    if not isinstance(df.columns, pd.MultiIndex):
        raise ValueError("Expected MultiIndex columns for DLC.")

    # find bodypart level
    bp_lvl = None
    for lvl in range(df.columns.nlevels):
        if bodypart in df.columns.get_level_values(lvl):
            bp_lvl = lvl
            break
    if bp_lvl is None:
        raise KeyError(f"Bodypart '{bodypart}' not found.")

    # slice to bodypart columns
    key = []
    for i in range(df.columns.nlevels):
        key.append(bodypart if i == bp_lvl else slice(None))
    bp = df.loc[:, pd.IndexSlice[tuple(key)]]

    # find coord level containing x/y
    coord_lvl = None
    for lvl in range(bp.columns.nlevels):
        vals = set(str(v).lower() for v in bp.columns.get_level_values(lvl))
        if "x" in vals and "y" in vals:
            coord_lvl = lvl
            break
    if coord_lvl is None:
        raise KeyError("Could not find coord level with x/y.")

    x = bp.xs("x", level=coord_lvl, axis=1).to_numpy().squeeze()
    y = bp.xs("y", level=coord_lvl, axis=1).to_numpy().squeeze()

    # likelihood is optional
    conf = None
    for lvl in range(bp.columns.nlevels):
        vals = set(str(v).lower() for v in bp.columns.get_level_values(lvl))
        if "likelihood" in vals or "p" in vals:
            # try likelihood first
            if "likelihood" in vals:
                conf = bp.xs("likelihood", level=lvl, axis=1).to_numpy().squeeze()
            else:
                conf = bp.xs("p", level=lvl, axis=1).to_numpy().squeeze()
            break

    return x, y, conf
def draw_trail(frame, pts, stride=2):
    """
    Draw trail as polyline. pts: list of (x,y) tuples.
    """
    if len(pts) < 2:
        return
    pts2 = pts[::max(1, int(stride))]
    arr = np.array(pts2, dtype=np.int32).reshape(-1, 1, 2)
    cv2.polylines(frame, [arr], isClosed=False, color=(0, 250, 0), thickness=2)

h5_files = sorted(glob.glob(os.path.join(DLC_DIR, "*filtered.h5")))
print(f"Found {len(h5_files)} h5 files")
for h5_path in h5_files:
    print("\nH5:", os.path.basename(h5_path))

    vid_path = find_matching_video(h5_path, VIDEO_DIR)
    if vid_path is None:
        print("  [WARN] No matching video found. Skipping.")
        continue
    print("  Video:", os.path.basename(vid_path))

    # load tracking
    df = pd.read_hdf(h5_path)
    x, y, conf = extract_xy_conf(df, BODYPART)

    cap = cv2.VideoCapture(vid_path)
    if not cap.isOpened():
        print("  [WARN] Could not open video. Skipping.")
        continue

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # output
    out_name = os.path.splitext(os.path.basename(vid_path))[0] + f"_{BODYPART}_overlay.mp4"
    out_path = os.path.join(OUT_DIR, out_name)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (W, H))

    trail_pts = []

    # ensure same length
    T = min(len(x), n_frames)

    for i in range(T):
        ok, frame = cap.read()
        if not ok:
            break

        xi, yi = x[i], y[i]

        # if likelihood exists, drop low confidence
        if conf is not None and CONF_THRESH is not None:
            if not np.isfinite(conf[i]) or conf[i] < CONF_THRESH:
                writer.write(frame)
                continue

        if np.isfinite(xi) and np.isfinite(yi):
            px = int(np.clip(xi, 0, W - 1))
            py = int(np.clip(yi, 0, H - 1))

            trail_pts.append((px, py))
            if TRAIL_MAXLEN is not None and len(trail_pts) > int(TRAIL_MAXLEN):
                trail_pts = trail_pts[-int(TRAIL_MAXLEN):]

            if TRAIL:
                draw_trail(frame, trail_pts, stride=TRAIL_STRIDE)

            # point
            cv2.circle(frame, (px, py), RADIUS, (0, 0, 255), -1)

        # label
        cv2.putText(frame, f"{BODYPART}  frame {i+1}/{T}", (15, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        writer.write(frame)

    cap.release()
    writer.release()

    print("  Saved overlay:", out_path)

print("\nDone.")

Found 148 h5 files

H5: 105-1Test 2-cliped_sDLC_DekrW32_3chamDec3shuffle1_snapshot_200_filtered.h5
  Video: 105-1Test 2-cliped_s.mp4
  Saved overlay: /Users/annateruel/Desktop/interpolated/overlay_videos/105-1Test 2-cliped_s_snout_overlay.mp4

H5: 105-2Test-4-cliped_2DLC_DekrW32_3chamDec3shuffle1_snapshot_200_filtered.h5
  Video: 105-2Test-4-cliped_2.mp4
  Saved overlay: /Users/annateruel/Desktop/interpolated/overlay_videos/105-2Test-4-cliped_2_snout_overlay.mp4

H5: 12-11-23_Exp7SalineCtrl_d211101-2-cliped_v2DLC_DekrW32_3chamDec3shuffle1_snapshot_200_filtered.h5
  Video: 12-11-23_Exp7SalineCtrl_d211101-2-cliped_v2.mp4
  Saved overlay: /Users/annateruel/Desktop/interpolated/overlay_videos/12-11-23_Exp7SalineCtrl_d211101-2-cliped_v2_snout_overlay.mp4

H5: 12-11-23_Exp7SalineCtrl_d211103-4-cliped_v2DLC_DekrW32_3chamDec3shuffle1_snapshot_200_filtered.h5
  Video: 12-11-23_Exp7SalineCtrl_d211103-4-cliped_v2.mp4
  Saved overlay: /Users/annateruel/Desktop/interpolated/overlay_videos/12-11-23_

For visualization porpuse, flip the videos

In [ ]:
import os
import shutil
import cv2
import pandas as pd

INDEX_CSV = '/Users/annateruel/Desktop/wanhui/index.csv'
VIDEO_IN_DIR  = '/Users/annateruel/Desktop/raw_flipped'
VIDEO_OUT_DIR = '/Users/annateruel/Desktop/wanhui/raw_flipped'
VIDEO_EXT      = '.mp4'
TARGET_NA_SIDE = 'L'  

os.makedirs(VIDEO_OUT_DIR, exist_ok=True)

def extract_base_name(fullname: str) -> str:
    """
    Remove DLC / snapshot suffix from index 'name' to match .mp4 filenames.

    Example:
      'ms5302Test_12-clipedDLC_DekrW32_3chamDec3shuffle1_snapshot_200_filtered'
        -> 'ms5302Test_12-cliped'
    """
    markers = ["DLC_DekrW32", "DLC", "sDLC", "snapshot"]
    base = fullname
    for m in markers:
        if m in base:
            base = base.split(m)[0]
    return base.rstrip("_- ")
def flip_video_lr(in_path, out_path):
    """
    Open a video, flip each frame left-right, and write to out_path.
    """
    cap = cv2.VideoCapture(in_path)
    if not cap.isOpened():
        print(f"[ERROR] Could not open video: {in_path}")
        return False

    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if fps <= 0:
        fps = 30  # fallback

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"  Flipping {n_frames} frames → {os.path.basename(out_path)}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        flipped = cv2.flip(frame, 1)  # horizontal flip
        writer.write(flipped)

    cap.release()
    writer.release()
    return True
def main():
    info = pd.read_csv(INDEX_CSV)

    for _, row in info.iterrows():
        name    = str(row["name"])
        na_side = str(row["NA"]).strip().upper()

        base = extract_base_name(name)   # e.g. "ms5302Test_12-cliped"
        in_path  = os.path.join(VIDEO_IN_DIR, base + VIDEO_EXT)
        out_path = os.path.join(VIDEO_OUT_DIR, base + VIDEO_EXT)

        if not os.path.exists(in_path):
            print(f"[WARN] Video not found for base '{base}', skipping: {in_path}")
            continue

        if os.path.exists(out_path):
            print(f"[SKIP] Output already exists, skipping: {out_path}")
            continue

        print(f"\nProcessing {in_path} | NA = {na_side}")

        if na_side == TARGET_NA_SIDE:
            # Already the orientation we want → copy
            shutil.copy2(in_path, out_path)
            print(f"  Copied (no flip) → {out_path}")
        else:
            # Flip to match TARGET_NA_SIDE
            ok = flip_video_lr(in_path, out_path)
            if ok:
                print(f"  Flipped and saved → {out_path}")
            else:
                print(f"  [ERROR] Failed to flip: {in_path}")


if __name__ == "__main__":
    main()